# Day 2 — Regression Models & Optimization

**Agri-AI Bootcamp** · Phase 1 of 5: *ML Fundamentals* · Days 1–4

> Build NumPy linear regression from scratch: manual gradient loop, MSE tracking, Matplotlib loss curve.

**Runtime:** a free CPU runtime is enough for today.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Linear regression fits a line y = w·x + b. MSE measures how far predictions sit from reality. Gradient descent nudges w and b downhill on the MSE surface; the learning rate is the size of each nudge.

### 🌾 Agriculture example

Nitrogen (kg/ha) → yield. If the learned slope is 0.02 t/ha per kg, adding 50 kg N is expected to add about 1 t/ha, within the range of your data.

> **Watch out:** Too high a learning rate makes the loss bounce or explode; too low crawls. Scale features first.

## 2. Setup

In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6, 3.5)

## 3. Data

In [ ]:
# ---- Farm-season data --------------------------------------------------------
# Course dataset: Kaggle "Crop Yield Prediction".  Kaggle needs an API token, so
# we simulate 200 farm-seasons with the same columns.  Replace this cell with
# pd.read_csv("yield_df.csv") once you have downloaded the real file.
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 200
year      = rng.integers(2005, 2025, n)
rainfall  = rng.normal(850, 180, n).clip(300, 1500)      # mm / season
fertiliser= rng.normal(120,  35, n).clip(20, 260)        # kg N / ha
# "true" relationship + noise, plus one drought year that behaves differently
yield_t   = (0.9 + 0.0022 * rainfall + 0.020 * fertiliser
             - 0.0000012 * rainfall ** 2 + rng.normal(0, 0.45, n))
yield_t[year == 2012] -= 1.8                             # drought season
df = pd.DataFrame(dict(year=year, rainfall=rainfall,
                       fertiliser=fertiliser, yield_t_ha=yield_t.clip(0.2, None)))
X = df[["rainfall", "fertiliser", "year"]].to_numpy(float)
y = df["yield_t_ha"].to_numpy(float)
df.head()

# today we use nitrogen only: one feature, so the line is easy to picture
x_raw = df["fertiliser"].to_numpy(float)
y_t   = df["yield_t_ha"].to_numpy(float)
x_mu, x_sd = x_raw.mean(), x_raw.std()
x = (x_raw - x_mu) / x_sd            # scaled -> gradient descent behaves
print("x scaled: mean %.2f std %.2f" % (x.mean(), x.std()))

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
w=b=0.0; lr=0.01
for e in range(500):
    err=w*x+b-y
    w-=lr*2*(err*x).mean(); b-=lr*2*err.mean()
    loss=(err**2).mean()   # store and plot
```

## 5. Worked solution

In [ ]:
# ---- Linear regression by hand: y = w*x + b, trained with gradient descent ---
def fit(x, y, lr=0.05, epochs=500):
    w = b = 0.0
    hist = []
    for _ in range(epochs):
        err  = w * x + b - y                 # d MSE / d pred
        grad_w = 2 * (err * x).mean()
        grad_b = 2 * err.mean()
        w -= lr * grad_w
        b -= lr * grad_b
        hist.append((err ** 2).mean())
    return w, b, np.array(hist)

w, b, hist = fit(x, y_t)
print(f"scaled space: w={w:.3f} b={b:.3f} | final MSE {hist[-1]:.4f}")

# un-scale so the slope is in real units: t/ha per kg N
slope = w / x_sd
intercept = b - w * x_mu / x_sd
print(f"real units  : {slope:.4f} t/ha per kg N  (intercept {intercept:.2f})")
print(f"=> +50 kg N  ~ +{50 * slope:.2f} t/ha, INSIDE the observed "
      f"{x_raw.min():.0f}-{x_raw.max():.0f} kg range only")

# closed-form check (normal equation) -- should match
A = np.c_[x, np.ones_like(x)]
w_ls, b_ls = np.linalg.lstsq(A, y_t, rcond=None)[0]
print(f"least squares: w={w_ls:.3f} b={b_ls:.3f}")

# ---- loss curve + fitted line ------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist); ax[0].set(xlabel="epoch", ylabel="MSE", title="loss curve")
ax[1].scatter(x_raw, y_t, s=12, alpha=.5, label="farm-seasons")
xs = np.linspace(x_raw.min(), x_raw.max(), 50)
ax[1].plot(xs, intercept + slope * xs, lw=2, color="crimson", label="fit")
ax[1].set(xlabel="nitrogen kg/ha", ylabel="yield t/ha"); ax[1].legend()
plt.tight_layout(); plt.show()

# ---- learning-rate sweep: too big explodes, too small crawls -----------------
for lr in [1.5, 0.3, 0.05, 0.001]:
    _, _, h = fit(x, y_t, lr=lr, epochs=200)
    tag = "diverged" if not np.isfinite(h[-1]) or h[-1] > h[0] else f"MSE {h[-1]:.4f}"
    print(f"lr={lr:<6} -> {tag}")

## 6. Your turn

1. Re-run `fit` on the UNSCALED `x_raw` with lr=0.05. What happens, and why does scaling fix it?
2. Add rainfall as a second feature: make w a length-2 vector and use `X @ w + b`.
3. Track w and b each epoch and plot their paths — you are watching the descent.

In [ ]:
# TODO 1: unscaled input (expect overflow / nan -- that is the lesson)
# w2, b2, h2 = fit(x_raw, y_t, lr=0.05, epochs=50); print(h2[:5])

# TODO 2: two features -- rainfall + nitrogen
Xm = df[["rainfall", "fertiliser"]].to_numpy(float)
Xm = (Xm - Xm.mean(0)) / Xm.std(0)

def fit_multi(X, y, lr=0.05, epochs=800):
    w = np.zeros(X.shape[1]); b = 0.0; hist = []
    for _ in range(epochs):
        err = X @ w + b - y
        w -= lr * 2 * (X.T @ err) / len(y)
        b -= lr * 2 * err.mean()
        hist.append((err ** 2).mean())
    return w, b, np.array(hist)

w_m, b_m, h_m = fit_multi(Xm, y_t)
print("two-feature w:", w_m.round(3), "b:", round(b_m, 3), "MSE:", round(h_m[-1], 4))
print("one feature MSE was", round(hist[-1], 4), "-- more features, lower training MSE")

# TODO 3: plot the (w, b) trajectory yourself

## 7. Dataset and references

**Dataset:** [Kaggle: Crop Yield Prediction](https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset)

**References**

- [Stanford CS229 notes](https://cs229.stanford.edu)
- [3Blue1Brown neural nets](https://www.3blue1brown.com/topics/neural-networks)

**Course material:** Andrew Ng ML Notes; 3Blue1Brown Gradient Descent

---
**Next:** Day 3 — Classification Fundamentals